**Starting With Combined Media (Twitter & Article Signals)**

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_absolute_error
from statsmodels.tsa.stattools import adfuller
import re
import ast
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch.utils.data as data
import random
import copy
import torch
from sklearn.model_selection import TimeSeriesSplit
import torch.nn as nn

In [37]:
def adf_test(series, name):

    result = adfuller(series)

    print("="*60)
    print(name)
    print(f"ADF Statistic : {result[0]:.4f}")
    print(f"P-value       : {result[1]:.4f}")

    if result[1] < 0.05:    
        print("Stationary")
    else:
        print("Non-stationary")

In [38]:
California= pd.read_csv('../data/interim/Media_with_features.csv')

In [39]:
California.head()

,year_month,article_count,rate_per_100k,mean_title_sentiment,mean_title_risk,mean_body_risk,avg_sentiment,avg_risk,mean_body_sentiment,tweet_count,avg_likes
0,2011-02,1.0,0.715017,0.028871,0.000000,0.015097,-0.465573,0.047619,-0.273456,6,0.000000
1,2011-03,1.0,0.821057,0.021148,0.000000,0.005263,-0.012873,0.000000,0.198157,5,1.000000
2,2011-04,0.0,0.833176,-0.125413,0.036783,0.020489,-0.185547,0.021802,-0.167308,0,7.516662
3,2011-05,0.0,0.875592,-0.125413,0.036783,0.020489,-0.185547,0.021802,-0.167308,0,7.516662
4,2011-06,0.0,0.969514,-0.125413,0.036783,0.020489,-0.185547,0.021802,-0.167308,0,7.516662


In [40]:
California.columns

Index(['year_month', 'article_count', 'rate_per_100k', 'mean_title_sentiment',
       'mean_title_risk', 'mean_body_risk', 'avg_sentiment', 'avg_risk',
       'mean_body_sentiment', 'tweet_count', 'avg_likes'],
      dtype='str')

In [41]:
cols =['article_count', 'rate_per_100k', 'mean_title_sentiment',
       'mean_title_risk', 'mean_body_risk', 'avg_sentiment', 'avg_risk',
       'mean_body_sentiment', 'tweet_count', 'avg_likes']

for col in cols:
    adf_test(California[col], col)

adf_test(California['rate_per_100k'], "VF Rate per 100k")

article_count
ADF Statistic : -7.0902
P-value       : 0.0000
Stationary
rate_per_100k
ADF Statistic : -0.2003
P-value       : 0.9385
Non-stationary
mean_title_sentiment
ADF Statistic : -11.0878
P-value       : 0.0000
Stationary
mean_title_risk
ADF Statistic : -12.3249
P-value       : 0.0000
Stationary
mean_body_risk
ADF Statistic : -10.8205
P-value       : 0.0000
Stationary
avg_sentiment
ADF Statistic : -4.3870
P-value       : 0.0003
Stationary
avg_risk
ADF Statistic : -3.4830
P-value       : 0.0084
Stationary
mean_body_sentiment
ADF Statistic : -13.7563
P-value       : 0.0000
Stationary
tweet_count
ADF Statistic : -2.3188
P-value       : 0.1660
Non-stationary
avg_likes
ADF Statistic : -2.4387
P-value       : 0.1311
Non-stationary
VF Rate per 100k
ADF Statistic : -0.2003
P-value       : 0.9385
Non-stationary


In [1]:
California['log_rate'] = np.log1p(California['rate_per_100k'])

NameError: name 'np' is not defined

In [44]:
California
24-11
13*12
# 155, 153 150

156

In [45]:
threem_test_size= 153
onem_test_size = 155
sixm_test_size=150

In [46]:
train1m = California.iloc[:-onem_test_size]
test1m = California.iloc[-onem_test_size:]

train3m = California.iloc[:-threem_test_size]
test3m = California.iloc[-threem_test_size:]

train6m = California.iloc[:-sixm_test_size]
test6m = California.iloc[-sixm_test_size:]

In [47]:
def create_dataset(dataset, lookback):
    X, y = [], []

    for i in range(len(dataset) - lookback):
        
        # Past log_rate + exogenous variables
        X.append(
            dataset[i:i+lookback, :]
        )
        
        # Next log_rate value (column 0)
        y.append(
            dataset[i+lookback, 0]
        )

    return (
        torch.tensor(
            np.array(X),
            dtype=torch.float32
        ),
        torch.tensor(
            np.array(y),
            dtype=torch.float32
        ).reshape(-1, 1)
    )

In [48]:
class LSTMModel(nn.Module):
    def __init__(
        self,
        n_features,
        hidden_size=50,
        num_layers=1,
        dropout=0
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.linear = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.linear(x)

        return x

In [49]:

tscv = TimeSeriesSplit(
    n_splits=5,
    test_size=12
)

In [50]:
results = []

for lookback in [3, 6, 12]:
    for hidden_size in [16, 32, 64]:
        for num_layers in [1, 2]:  
            for lr in [0.001, 0.0001]:  
                for batch_size in [8, 16]: 
                    for train_idx, val_idx in tscv.split(train1m): 
                        train_fold = train1m.iloc[train_idx] 
                        val_fold = train1m.iloc[val_idx] 

                        # -----------------------------
                        # 1. SCALE INSIDE EACH FOLD
                        # -----------------------------

                        scaler = MinMaxScaler()
                        train_scaled = scaler.fit_transform(
                            train_fold)
                        val_scaled = scaler.transform(val_fold)


                        # -----------------------------
                        # 2. CREATE SEQUENCES
                        # -----------------------------

                        X_train, y_train = create_dataset(
                            train_scaled,
                            lookback
                        )


                        # For validation, include previous lookback months
                        # from training history

                        val_input = np.concatenate(
                            [train_scaled[-lookback:], val_scaled], axis=0)

                        X_val, y_val = create_dataset(val_input, lookback)

                        # Convert targets to correct shape
                        y_train = y_train.reshape(-1, 1)
                        y_val = y_val.reshape(-1, 1)

                        # -----------------------------
                        # 3. CONVERT TO PYTORCH TENSORS
                        # -----------------------------

                        X_train = X_train.float()
                        y_train = y_train.float()

                        X_val = X_val.float()
                        y_val = y_val.float()


                        # -----------------------------
                        # 4. CREATE MODEL FOR THIS FOLD
                        # -----------------------------

                        model = LSTMModel(
                            n_features=X_train.shape[2],
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=0.6)

                        optimizer = torch.optim.Adam(
                            model.parameters(),
                            lr=lr
                        )

                        loss_fn = nn.MSELoss()

                        # -----------------------------
                        # 5. CREATE DATALOADER
                        # -----------------------------

                        train_loader = data.DataLoader(data.TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=False)


                        # -----------------------------
                        # 6. TRAIN MODEL
                        # -----------------------------

                        best_val_rmse = np.inf
                        patience = 20
                        patience_counter = 0
                        best_epoch = 0

                        max_epochs = 300

                        for epoch in range(max_epochs):
                            model.train()
                            for X_batch, y_batch in train_loader:
                                y_pred = model(X_batch)
                                loss = loss_fn(y_pred,y_batch)

                                optimizer.zero_grad()
                                loss.backward()
                                optimizer.step()

                            # -------------------------
                            # VALIDATION RMSE
                            # -------------------------

                            model.eval()

                            with torch.no_grad():
                                val_pred = model(X_val)
                                val_rmse = torch.sqrt(torch.mean((val_pred - y_val) ** 2)).item()


                            # Early stopping

                            if val_rmse < best_val_rmse:
                                best_val_rmse = val_rmse
                                patience_counter = 0
                                best_epoch = epoch + 1
                                best_weights = copy.deepcopy(model.state_dict())
                            else:
                                patience_counter += 1

                            if patience_counter >= patience:
                                break

                        # Restore best model for this fold

                        model.load_state_dict(best_weights)


                        # -----------------------------
                        # 7. SAVE FOLD SCORE
                        # -----------------------------
                        results.append(
                        {"lookback": lookback,
                                    "hidden_size": hidden_size,
                                    "num_layers": num_layers,
                                    "lr": lr,
                                    "batch_size": batch_size,
                                    "best_epoch": best_epoch,
                                    "rmse": best_val_rmse,}
                            )

                        print(
                                f"Lookback={lookback}, Hidden={hidden_size}, Layers={num_layers}, "
                                f"LR={lr}, Batch={batch_size}, Best Epoch={best_epoch} | "
                                f"Fold RMSE={best_val_rmse:.4f}"
                            )

ValueError: Too many splits=5 for number of samples=12 with test_size=12 and gap=0.